# DSCD Hero Figures

This notebook regenerates the DSCD hero figures from a timestamped run folder under `output-dsfb-dscd/YYYYMMDD_HHMMSS/`.

In [ ]:
from pathlib import Path
import subprocess

RUN_DIR = None  # e.g. Path('output-dsfb-dscd/20260303_135003')
OUTPUT_ROOT = Path('output-dsfb-dscd')
DEMO_MODE = True
PERFORMANCE_MODE = False
AUTO_GENERATE = False  # Set True to generate a fresh run from this notebook
EDGE_PREVIEW_LIMIT = 512

if DEMO_MODE and PERFORMANCE_MODE:
    raise ValueError('Choose either DEMO_MODE or PERFORMANCE_MODE, not both.')
mode = 'performance' if PERFORMANCE_MODE else 'demo'

if AUTO_GENERATE:
    cmd = ['cargo', 'run', '-p', 'dsfb-dscd', '--release', '--', '--mode', mode]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if RUN_DIR is None:
    runs = sorted([path for path in OUTPUT_ROOT.glob('*') if path.is_dir()])
    if not runs:
        raise FileNotFoundError('No DSCD run folders found under output-dsfb-dscd/.')
    RUN_DIR = runs[-1]

RUN_DIR = Path(RUN_DIR).expanduser().resolve()
print('Mode:', mode)
print('Using RUN_DIR:', RUN_DIR)

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

summary = pd.read_csv(RUN_DIR / 'threshold_scaling_summary.csv')
curve_files = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))
if not curve_files:
    raise FileNotFoundError('No threshold_curve_N_*.csv files found in RUN_DIR')

curve_frames = []
for file in curve_files:
    n = int(file.stem.split('_')[-1])
    curve = pd.read_csv(file)
    curve['num_events'] = n
    curve_frames.append(curve)
curves = pd.concat(curve_frames, ignore_index=True)

events_df = pd.read_csv(RUN_DIR / 'graph_events.csv')
edges_df = pd.read_csv(RUN_DIR / 'graph_edges.csv')

def resolve_edge_columns(df):
    candidates = [
        ('src', 'dst'),
        ('source_event_id', 'target_event_id'),
        ('from_event_id', 'to_event_id'),
        ('source', 'target'),
    ]
    for src_col, dst_col in candidates:
        if src_col in df.columns and dst_col in df.columns:
            return src_col, dst_col
    raise KeyError(f'Could not resolve edge columns from: {list(df.columns)}')

src_col, dst_col = resolve_edge_columns(edges_df)

prov_candidates = [RUN_DIR / 'edge_provenance_0.csv', RUN_DIR / 'edge_provenance.csv']
provenance_path = next((p for p in prov_candidates if p.exists()), None)
if provenance_path is None:
    raise FileNotFoundError('No edge_provenance_0.csv or edge_provenance.csv found')
provenance_df = pd.read_csv(provenance_path)

print('Loaded curves for N:', sorted(curves['num_events'].unique()))
print('Loaded provenance from:', provenance_path.name)

In [ ]:
# Figure 1: Hero rho(tau) for largest N with tau* marker
largest_n = int(curves['num_events'].max())
hero = curves[curves['num_events'] == largest_n].sort_values('tau')

summary_row = summary[summary['num_events'] == largest_n]
tau_star = float(summary_row['tau_star'].iloc[0]) if not summary_row.empty else np.nan

plt.figure(figsize=(8, 5))
plt.plot(hero['tau'], hero['expansion_ratio'], linewidth=2.5, label=f'N={largest_n}')
if np.isfinite(tau_star):
    plt.axvline(tau_star, color='tab:red', linestyle='--', label=f'tau*={tau_star:.4f}')
plt.xlabel('tau')
plt.ylabel('rho(tau)')
plt.title('DSCD Hero: Expansion Ratio vs Trust Threshold')
plt.grid(alpha=0.25)
plt.legend()
plt.savefig(RUN_DIR / 'fig_dscd_hero_rho_vs_tau.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: finite-size scaling
summary = summary.sort_values('num_events')
N = summary['num_events'].to_numpy(dtype=float)
width = summary['transition_width'].to_numpy(dtype=float)
max_derivative = summary['max_derivative'].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].loglog(N, np.maximum(width, 1e-12), marker='o', linewidth=2)
axes[0].set_title('Transition Width vs N')
axes[0].set_xlabel('num_events')
axes[0].set_ylabel('transition_width')
axes[0].grid(alpha=0.25, which='both')

axes[1].loglog(N, np.maximum(max_derivative, 1e-12), marker='o', linewidth=2)
axes[1].set_title('Max Derivative vs N')
axes[1].set_xlabel('num_events')
axes[1].set_ylabel('max_derivative')
axes[1].grid(alpha=0.25, which='both')

plt.tight_layout()
plt.savefig(RUN_DIR / 'fig_dscd_finite_size_scaling.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: local DAG preview
def resolve_event_columns(df):
    id_col = next((c for c in ['event_id', 'id'] if c in df.columns), None)
    if id_col is None:
        raise KeyError(f'Could not resolve event id column from: {list(df.columns)}')

    time_col = next((c for c in ['time_index', 'time', 'event_index', 't'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Could not resolve time column from: {list(df.columns)}')

    structural_col = next((c for c in ['structural_level', 'entropy_density', 'echo_slope'] if c in df.columns), None)
    return id_col, time_col, structural_col


event_id_col, time_col, structural_col = resolve_event_columns(events_df)
events_sorted = events_df.sort_values(time_col).reset_index(drop=True)

window_events = min(max(128, EDGE_PREVIEW_LIMIT // 2), len(events_sorted))
start = max(0, (len(events_sorted) // 2) - (window_events // 2))
window_df = events_sorted.iloc[start:start + window_events]
window_ids = set(window_df[event_id_col])

local_edges = edges_df[
    edges_df[src_col].isin(window_ids) & edges_df[dst_col].isin(window_ids)
].copy()

time_map = dict(zip(events_df[event_id_col], events_df[time_col]))
if 'temporal_lag' in local_edges.columns:
    local_edges = local_edges.sort_values('temporal_lag')
else:
    local_edges['temporal_lag_plot'] = (
        local_edges[dst_col].map(time_map) - local_edges[src_col].map(time_map)
    ).abs()
    local_edges = local_edges.sort_values('temporal_lag_plot')

subset_edges = local_edges.head(EDGE_PREVIEW_LIMIT)
if subset_edges.empty:
    fallback_edges = edges_df.copy()
    if 'temporal_lag' in fallback_edges.columns:
        fallback_edges = fallback_edges.sort_values('temporal_lag')
    else:
        fallback_edges['temporal_lag_plot'] = (
            fallback_edges[dst_col].map(time_map) - fallback_edges[src_col].map(time_map)
        ).abs()
        fallback_edges = fallback_edges.sort_values('temporal_lag_plot')
    subset_edges = fallback_edges.head(EDGE_PREVIEW_LIMIT)

subset_nodes = sorted(set(subset_edges[src_col]).union(subset_edges[dst_col]))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[[src_col, dst_col]].itertuples(index=False, name=None))

level_map = (
    dict(zip(events_df[event_id_col], events_df[structural_col]))
    if structural_col is not None
    else {}
)

x_raw = np.array([float(time_map.get(node, 0.0)) for node in graph.nodes()], dtype=float)
if len(x_raw) and np.max(x_raw) > np.min(x_raw):
    x = (x_raw - np.min(x_raw)) / (np.max(x_raw) - np.min(x_raw))
else:
    x = np.zeros_like(x_raw)

if structural_col is not None:
    y_raw = np.array([float(level_map.get(node, 0.0)) for node in graph.nodes()], dtype=float)
    if len(y_raw) and np.max(y_raw) > np.min(y_raw):
        y = (y_raw - np.min(y_raw)) / (np.max(y_raw) - np.min(y_raw))
    else:
        y = np.zeros_like(y_raw)
else:
    order = {node: i for i, node in enumerate(sorted(graph.nodes()))}
    y = np.array([((order[node] % 17) - 8) / 16.0 for node in graph.nodes()], dtype=float)

order = {node: i for i, node in enumerate(sorted(graph.nodes()))}
jitter = np.array([((order[node] % 9) - 4) * 0.01 for node in graph.nodes()], dtype=float)
y = y + jitter

pos = {node: (float(xi), float(yi)) for node, xi, yi in zip(graph.nodes(), x, y)}

plt.figure(figsize=(10, 6))
nx.draw_networkx_edges(graph, pos=pos, alpha=0.35, width=0.7, arrows=True, arrowsize=10)
nx.draw_networkx_nodes(graph, pos=pos, node_size=55, node_color='tab:blue')
plt.title(f'DSCD DAG preview ({len(subset_nodes)} nodes, {len(subset_edges)} local edges)')
plt.xlabel('normalized time')
plt.ylabel(structural_col if structural_col is not None else 'layout rank')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RUN_DIR / 'fig_dscd_dag_preview.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 4: provenance panel
row = provenance_df.iloc[0]
source_id = int(row['source_event']) if 'source_event' in provenance_df.columns else int(row.get('src', row.get('source_event_id', 0)))
target_id = int(row['target_event']) if 'target_event' in provenance_df.columns else int(row.get('dst', row.get('target_event_id', 0)))

sub = edges_df[((edges_df[src_col] == source_id) | (edges_df[src_col] == target_id) |
                (edges_df[dst_col] == source_id) | (edges_df[dst_col] == target_id))].head(128)
sub_nodes = sorted(set(sub[src_col]).union(sub[dst_col]))

pg = nx.DiGraph()
pg.add_nodes_from(sub_nodes)
pg.add_edges_from(sub[[src_col, dst_col]].itertuples(index=False, name=None))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pos = nx.spring_layout(pg, seed=5)
colors = ['tab:red' if node in (source_id, target_id) else 'tab:blue' for node in pg.nodes()]
sizes = [220 if node in (source_id, target_id) else 80 for node in pg.nodes()]
nx.draw_networkx(pg, pos=pos, node_color=colors, node_size=sizes, with_labels=False, arrows=True, ax=axes[0])
axes[0].set_title('Provenance edge neighborhood')
axes[0].axis('off')

axes[1].axis('off')
keys = [
    'edge_id', 'source_event', 'target_event', 'tau_star', 'base_trust', 'effective_trust',
    'dt', 'd_struct', 'structural_level_source', 'structural_level_target',
    'module_id', 'rewrite_rule_id', 'residual_state_source', 'residual_state_target',
]
lines = []
for key in keys:
    if key in provenance_df.columns:
        lines.append(f'{key}: {row[key]}')
axes[1].text(0.02, 0.98, '\n'.join(lines), va='top', family='monospace')
axes[1].set_title('Edge provenance')

plt.tight_layout()
plt.savefig(RUN_DIR / 'fig_dscd_edge_provenance.png', dpi=300, bbox_inches='tight')
plt.show()